In [ ]:
from pyscf import gto, scf
from pyEDA import * 
from utils import make_frag_geom, index_parse

In [ ]:
import numpy as np
np.set_printoptions(formatter={"float": "{: 0.3f}".format})

To perform Energy Decomposition Analysis (EDA), we first need to run SCF calculations for the molecule and its fragments using ghost atoms.

To build fragments with ghost atoms, you can use the `make_frag_geom` function. Its inputs are a `gto.Mole` object for the full molecule and a list of atom indices that should be marked as ghosts.

Using `unit="BOHR"` for the fragments is essential because `make_frag_geom` relies on the internal molecular geometry representation of **PySCF**, whose length unit is Bohr, not angstroms.

After constructing the molecule and its fragments, we run SCF calculations and obtain mean-field objects (RKS or UKS). These objects are the inputs for the Energy Decomposition Analysis.

In [ ]:
from pyscf import gto, scf

# ─── Build molecules ─────────────────────────────────────────────────────────
def make_mol(atom_str, units = 'Ang'):
    mol = gto.Mole()
    mol.atom  = atom_str
    mol.spin  = 0
    mol.basis = BASIS
    mol.build(unit = units)
    return mol

def run_dft(mol):
    mf      = scf.RKS(mol)
    mf.xc   = METHOD
    mf.disp = DISP
    mf.run()
    return mf

# ─── Settings ────────────────────────────────────────────────────────────────
METHOD = "pbe0"
BASIS  = "def2-tzvp"
DISP   = "d3bj"

# ─── Geometry ────────────────────────────────────────────────────────────────
# BH3–NH3 dimer: A = BH3, B = NH3

AB_mol = make_mol('coords.xyz')

A_xyz = make_frag_geom(AB_mol, "0-3")
B_xyz = make_frag_geom(AB_mol, "4-8")

A_mol  = make_mol(A_xyz, units = 'BOHR')   
B_mol  = make_mol(B_xyz, units = 'BOHR')   

# ─── Run DFT ─────────────────────────────────────────────────────────────────

AB_mf = run_dft(AB_mol)
A_mf  = run_dft(A_mol)
B_mf  = run_dft(B_mol)

Now we can perform Energy Decomposition Analysis using the corresponding EDA class and its run method.

In [ ]:
eda = ETS_NOCV(AB_mf, [A_mf, B_mf])
eda.run()

The analysis results are stored in two dictionaries:

* `EDA`, which contains the energy terms (in Hartree),

* `NOCV`, which contains the orbital decomposition data.

In [ ]:
print(eda.EDA)

In [ ]:
print(eda.NOCV)

For visualization, you can use the `write_molden` function, which writes a Molden file for the NOCV orbitals. Of particular interest are the Pauli and orbital densities, as well as the contribution of the k-th NOCV pair to the deformation density.

To generate the Molden file, use `write_molden`. It will create a file named
`NOCV.molden` in the working directory.

To generate the Pauli and orbital densities, use the `gen_PauliOrb_densities` function. This will create the files
`pauli.cube` and `orb.cube`.

Finally, to generate the contribution of the k-th NOCV pair to the deformation density, use `gen_pair_density`. It will create a file named
`{k}_pair_density_restricted.cube`. Indexation begins with 0.

The last two functions accept the options `nx`, `ny`, `nz`, `resolution`, and `margin`, which control the grid size and spatial extent used during cube file generation.

In [ ]:
eda.write_molden()

In [ ]:
eda.gen_pair_density(0, nx = 100, ny = 100, nz = 100, margin = 3)

Now let see the deformation density, to do that we will use py3Dmol package.

In [ ]:
with open("coords.xyz", "r") as f:
    xyz_data = f.read()

view = py3Dmol.view(width=400, height=400)

view.addModel(xyz_data, "xyz")

view.setStyle({"sphere": {"radius": 0.3}, "stick": {"radius": 0.15}})

view.zoomTo()

view.show()

In [ ]:
def draw_orbital(cube, xyz):
    with open(cube) as f:
        cube_data = f.read()
    view.addVolumetricData(
        cube_data, "cube", {"isoval": -0.005, "color": "red", "opacity": 0.75}
    )
    view.addVolumetricData(
        cube_data, "cube", {"isoval": 0.005, "color": "blue", "opacity": 0.75}
    )
    view.addModel(xyz, "xyz")
    view.setStyle({"sphere": {"radius": 0.3}, "stick": {"radius": 0.15}})
    view.zoomTo()
    view.update()
    view.clear()

In [ ]:
view = py3Dmol.view(width=400, height=400)
view.show()
draw_orbital('0_pair_density_restricted.cube', xyz_data)